In [ ]:
import os
os.chdir('???')
os.getcwd()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Load dataset

In [ ]:
orig_df = pd.read_csv("bike_day.csv")
orig_df.head(10)

In [ ]:
df = orig_df.drop(["instant", "dteday"], axis=1)
df = df.copy()
orig_X = df.drop("cnt", axis=1)
X = orig_X.copy()

### Scaling

In [ ]:
# Scale by standardized normal distribution, (x-mean)/sd
def scale_standard(X_train, X_test):
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Scale by min-max, (x-min)/(max-min)
def scale_minmax(X_train, X_test):
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Scale by robust, (x-median)/iqr
def scale_minmax(X_train, X_test):
    from sklearn.preprocessing import RobustScaler
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

## Performance Evaluation

In [ ]:
def compute_regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

def compute_mape(y_true, y_pred):
    # Mean Absolute Percentage Error
    # Note: MAPE fails if y_true contains zeros.
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def compute_smape(y_true, y_pred):
    # Symmetric Mean Absolute Percentage Error
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

def compute_rmsle(y_true, y_pred, neg_mask=0):
    # Root Mean Squared Log Error
    # RMSLE is good for right-skewed data
    # Note: log1p(x) = log(1+x), and the log parameter must be positive. 
    # Therefore, x >= -1. Otherwise, log1p(1+x) encounters an invalid value.
        
    if neg_mask == 1:
        mask = (y_true >= 0) & (y_pred >= 0)
        y_true = y_true[mask]
        y_pred = y_pred[mask]
    
    np_y_true, np_y_pred = np.array(y_true), np.array(y_pred)
    rmsle = np.sqrt(np.mean((np.log1p(np_y_true) - np.log1p(np_y_pred))**2))  # Root Mean Squared Log Error

    return rmsle

### Experiments: Models

In [ ]:
# Train and test model
# Remove high-correlated variables with y (so the model does not perfectly predict responses.)
selected_vars = ['season', 'yr', 'mnth', 'holiday', 'weekday','workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
X_base = orig_X[selected_vars]
y_base = df["cnt"]

# Train/Test Split
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(X_base, y_base, test_size=0.2, random_state=42)

### Linear Regression

In [ ]:
X_train, y_train = X_train_base.copy(), y_train_base.copy()
X_test, y_test = X_test_base.copy(), y_test_base.copy()

######################################
# Construct a model
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
######################################

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

performance_arr = {}
performance_arr['Linear'] = [rmse, mae, r2, mape]

In [ ]:
model.coef_

### Regularized Regression

#### Ridge Regression

In [ ]:
X_train, y_train = X_train_base.copy(), y_train_base.copy()
X_test, y_test = X_test_base.copy(), y_test_base.copy()

######################################
# Construct a model
selected_alpha = 1.0   # alpha = λ

# Penalties in regularized regression are aadded to based on how large coefficients are.
# Thus, for regularized regression, scaling is needed.  
X_train, X_test = scale_standard(X_train, X_test)

from sklearn.linear_model import Ridge 
model = Ridge(alpha=selected_alpha)  
model.fit(X_train, y_train)
######################################

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

performance_arr['Ridge'] = [rmse, mae, r2, mape]

In [ ]:
model.coef_

#### Lasso Regression

In [ ]:
X_train, y_train = X_train_base.copy(), y_train_base.copy()
X_test, y_test = X_test_base.copy(), y_test_base.copy()

######################################
# Construct a model
selected_alpha = 0.1   # alpha = λ

# Penalties in regularized regression are aadded to based on how large coefficients are.
# Thus, for regularized regression, scaling is needed.  
X_train, X_test = scale_standard(X_train, X_test)

from sklearn.linear_model import Lasso
model = Lasso(alpha=selected_alpha)
model.fit(X_train, y_train)
######################################

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

performance_arr['Lasso'] = [rmse, mae, r2, mape]

In [ ]:
model.coef_

#### Elastic Net Regression

In [ ]:
X_train, y_train = X_train_base.copy(), y_train_base.copy()
X_test, y_test = X_test_base.copy(), y_test_base.copy()

######################################
# Construct a model
selected_alpha = 0.1   # alpha = λ
selected_l1_ratio=0.5  # l1_ratio=0.5 → 50% L1, 50% L2

# Penalties in regularized regression are aadded to based on how large coefficients are.
# Thus, for regularized regression, scaling is needed.  
X_train, X_test = scale_standard(X_train, X_test)

from sklearn.linear_model import ElasticNet
model = ElasticNet(alpha=selected_alpha, l1_ratio=selected_l1_ratio)  
model.fit(X_train, y_train)
######################################

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

performance_arr['Elastic net'] = [rmse, mae, r2, mape]

### KNN Regression

In [ ]:
X_train, y_train = X_train_base.copy(), y_train_base.copy()
X_test, y_test = X_test_base.copy(), y_test_base.copy()

######################################
# Construct a model
selected_k = 5

# knn regression depends on distance method.
# thus, to compute fair distance, scaling is needed.  
X_train, X_test = scale_standard(X_train, X_test)

from sklearn.neighbors import KNeighborsRegressor
model = KNeighborsRegressor(n_neighbors=selected_k)
model.fit(X_train, y_train)
######################################

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

performance_arr['KNN'] = [rmse, mae, r2, mape]

### Tree-based Regression

#### Decision Tree Regression

In [ ]:
X_train, y_train = X_train_base.copy(), y_train_base.copy()
X_test, y_test = X_test_base.copy(), y_test_base.copy()

######################################
# Construct a model
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor()
model.fit(X_train, y_train)
######################################

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

performance_arr['Decision Tree'] = [rmse, mae, r2, mape]

#### Random Forest Regression

In [ ]:
X_train, y_train = X_train_base.copy(), y_train_base.copy()
X_test, y_test = X_test_base.copy(), y_test_base.copy()

######################################
# Construct a model
selected_n_trees = 100      # Default = 100 = number of decision trees

# For more info, go to: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
from sklearn.ensemble import RandomForestRegressor
model= RandomForestRegressor(n_estimators=selected_n_trees)
model.fit(X_train, y_train)
######################################

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

performance_arr['Random Forest'] = [rmse, mae, r2, mape]

### Support Vector Regression

In [ ]:
X_train, y_train = X_train_base.copy(), y_train_base.copy()
X_test, y_test = X_test_base.copy(), y_test_base.copy()

######################################
# Construct a model
selected_kernel = 'rbf'         # Default = 'rbf'
selected_C = 1                  # Default = 1
selected_epsilon = 0.1          # Default = 0.1

from sklearn.svm import SVR
# For more info, go to: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html
model = SVR(kernel=selected_kernel, C=selected_C, epsilon=selected_epsilon)  # RBF kernel, adjust C and epsilon
model.fit(X_train, y_train)
######################################

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

performance_arr['SVR'] = [rmse, mae, r2, mape]

### Multiple-Layered Perceptron Regression

In [ ]:
X_train, y_train = X_train_base.copy(), y_train_base.copy()
X_test, y_test = X_test_base.copy(), y_test_base.copy()

######################################
# Construct a model
selected_layers = (10,5)         # Default = (100,)
selected_activate = 'relu'       # Default = 'relu', Options: {‘identity’, ‘logistic’, ‘tanh’, ‘relu’}
selected_max_iter = 10000        # Default = 200

# to find weights in mlp system, data values are important
# thus, to obtain mlp model, scaling is needed.  
X_train, X_test = scale_standard(X_train, X_test)

from sklearn.neural_network import MLPRegressor
# For more info, go to: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html
#model = MLPRegressor()
model = MLPRegressor(hidden_layer_sizes=selected_layers, activation=selected_activate, max_iter=selected_max_iter)
model.fit(X_train, y_train)
######################################

# Use model for predicted responses
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Measure performances
rmse, mae, r2 = compute_regression_metrics(y_test, y_pred)
mape = compute_mape(y_test, y_pred)
print("RMSE: ", rmse)
print("MAE: ", mae)
print("MAPE: ", mape)
print("R2: ", r2)
rmsle = compute_rmsle(y_test, y_pred)
print("RMSLE: ", rmsle)

performance_arr['MLP'] = [rmse, mae, r2, mape]

## Performance Comparison

In [ ]:
performance_arr

In [ ]:
metrics = ['MSE', 'RMSE', 'R2', 'MAE']
colors = {'MSE':'red', 'RMSE':'orange', 'R2':'blue', 'MAE':'purple'}

# Convert to DataFrame for easier plotting
df = pd.DataFrame(performance_arr, index=metrics)
df

In [ ]:
for metric in metrics:
    plt.figure(figsize=(10,4))
    df.loc[metric].plot(kind='bar',color=colors[metric])
    plt.title(f'{metric} Comparison Across Models')
    plt.ylabel(metric)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()